# 面试题：最终状态 Grader 怎样设计？

最终状态 grader 将用户目标编译为可由权威系统验证的谓词，而不是判断模型是否说了完成。退款任务要求金额匹配、订单状态成功、事件关联当前请求且无重复；超时则输出 pending/unknown。

## 真实案例

六个退款最终状态包括文本成功、金额不符、重复退款、pending 和完整成功。

## 基线

基线只检查 Agent 最后的完成文本。

## 结果解读

手写谓词 grader 输出缺失条件。

## 失败案例

文本说成功但退款账本金额不符，任务仍未完成。

In [1]:
states = [{'id':'F1','text':True,'status':'succeeded','amount':80,'expected':80,'request':'r1','event':'r1','duplicates':0}, {'id':'F2','text':True,'status':'succeeded','amount':50,'expected':80,'request':'r2','event':'r2','duplicates':0}, {'id':'F3','text':True,'status':'succeeded','amount':80,'expected':80,'request':'r3','event':'r3','duplicates':1}, {'id':'F4','text':False,'status':'pending','amount':0,'expected':80,'request':'r4','event':'r4','duplicates':0}, {'id':'F5','text':True,'status':'failed','amount':0,'expected':40,'request':'r5','event':'r5','duplicates':0}, {'id':'F6','text':True,'status':'succeeded','amount':30,'expected':30,'request':'r6','event':'r6','duplicates':0}]  # 构造六条从权威支付账本读取的最终状态。
print('最终状态输入:', states)  # 输出目标、账本和文本状态。
print('教学说明：status/amount/event 是权威系统读回，text 只是模型声明。')  # 区分事实与回答。

最终状态输入: [{'id': 'F1', 'text': True, 'status': 'succeeded', 'amount': 80, 'expected': 80, 'request': 'r1', 'event': 'r1', 'duplicates': 0}, {'id': 'F2', 'text': True, 'status': 'succeeded', 'amount': 50, 'expected': 80, 'request': 'r2', 'event': 'r2', 'duplicates': 0}, {'id': 'F3', 'text': True, 'status': 'succeeded', 'amount': 80, 'expected': 80, 'request': 'r3', 'event': 'r3', 'duplicates': 1}, {'id': 'F4', 'text': False, 'status': 'pending', 'amount': 0, 'expected': 80, 'request': 'r4', 'event': 'r4', 'duplicates': 0}, {'id': 'F5', 'text': True, 'status': 'failed', 'amount': 0, 'expected': 40, 'request': 'r5', 'event': 'r5', 'duplicates': 0}, {'id': 'F6', 'text': True, 'status': 'succeeded', 'amount': 30, 'expected': 30, 'request': 'r6', 'event': 'r6', 'duplicates': 0}]
教学说明：status/amount/event 是权威系统读回，text 只是模型声明。


In [2]:
baseline = [(row['id'], '完成' if row['text'] else '未完成') for row in states]  # 构造只看最终文本的基线。
print('文本基线:', baseline)  # 输出对 F2/F3 的错误完成。
print('基线问题：文字没有验证金额、关联与重复副作用。')  # 解释必须引入权威谓词。

文本基线: [('F1', '完成'), ('F2', '完成'), ('F3', '完成'), ('F4', '未完成'), ('F5', '完成'), ('F6', '完成')]
基线问题：文字没有验证金额、关联与重复副作用。


In [3]:
def grade_final(row):  # 定义基于权威账本的最终状态 grader。
    checks = {'status':row['status'] == 'succeeded','amount':row['amount'] == row['expected'],'correlation':row['event'] == row['request'],'unique':row['duplicates'] == 0}  # 计算四个业务完成谓词。
    if row['status'] == 'pending':  # 为最终一致或超时保留未知状态。
        return 'pending', checks  # 不把 pending 误判为成功或失败。
    return ('completed' if all(checks.values()) else 'incomplete'), checks  # 返回完成与缺失谓词。

In [4]:
results = [(row['id'],) + grade_final(row) for row in states]  # 对六个最终状态执行谓词评分。
print('id | 最终评分 | 谓词')  # 输出最终状态 grader 表标题。
for item in results:  # 遍历每个任务的完成证据。
    print(item[0], item[1], item[2])  # 输出状态、金额、关联和唯一性中间量。
print('真实完成数:', sum(item[1] == 'completed' for item in results))  # 汇总满足所有业务谓词的任务。

id | 最终评分 | 谓词
F1 completed {'status': True, 'amount': True, 'correlation': True, 'unique': True}
F2 incomplete {'status': True, 'amount': False, 'correlation': True, 'unique': True}
F3 incomplete {'status': True, 'amount': True, 'correlation': True, 'unique': False}
F4 pending {'status': False, 'amount': False, 'correlation': True, 'unique': True}
F5 incomplete {'status': False, 'amount': False, 'correlation': True, 'unique': True}
F6 completed {'status': True, 'amount': True, 'correlation': True, 'unique': True}
真实完成数: 2


In [5]:
wrong = dict(baseline)['F2']  # 读取金额不符但文本完成的基线结论。
fixed = dict((item[0], item[1]) for item in results)['F2']  # 读取权威谓词 grader 的结论。
print('失败案例 F2：文本=', wrong, '，最终状态=', fixed)  # 展示最终状态不能由模型口头声明。
print('生产差距：需隔离 fixture、目标编译、版本化 oracle、最终一致窗口和与 trace grader 的联合报告。')  # 说明发布级评测要素。

失败案例 F2：文本= 完成 ，最终状态= incomplete
生产差距：需隔离 fixture、目标编译、版本化 oracle、最终一致窗口和与 trace grader 的联合报告。


In [6]:
assert dict((item[0], item[1]) for item in results)['F1'] == 'completed'  # 验证完整账本状态通过。
assert dict((item[0], item[1]) for item in results)['F2'] == 'incomplete'  # 验证金额不符不能完成。
assert dict((item[0], item[1]) for item in results)['F4'] == 'pending'  # 验证 pending 保持未知状态。